> **Privacy note.** Raw videos are owner-only; only blurred frames are released. Private paths and clip file names are replaced by placeholders. All image outputs are removed from this copy.

# Al-Hada frames: extract, blur plates and people, save for the team
AI251P Capstone, Track 1 (dataset bonus). Owner: Abdulmohsen Alharthi.

**What this notebook does**
1. Reads the two original clips from the **private** folder `<owner-only private folder>` (owner only).
2. Extracts about 1 frame per second with ffmpeg (iPhone rotation handled, HDR tone-mapped when possible). Raw frames stay on the Colab local disk only.
3. Detects vehicles and people with a COCO-pretrained YOLO model, then blurs **every person box** and the **plate zone** of every vehicle large enough for a plate to be readable.
4. Saves **only the blurred frames** to `01_Data/Team_AlHada/Frames` (shared with the team), plus a manifest CSV and a summary JSON.
5. Shows a review sheet. Check it by eye. If anything is still readable, add a box in `MANUAL_BLUR` and re-run the blur cells.

**Rules:** the raw clips and unblurred frames never leave the private folder or the Colab disk. Frames carry no GPS (ffmpeg does not copy EXIF). Al-Hada is a **test-only** domain: never used for training or model selection.

**Runtime:** T4 GPU recommended (Runtime > Change runtime type). Run with the KAUST account (owner of the private folder).

## 0. Settings

In [1]:
# ---- edit these if needed ----
RAW_DIR = '<owner-only private folder>'
CLIPS = {'CLIP01': '<clip 1>',    # wide view, 2:44
         'CLIP02': '<clip 2>'}    # 2x zoom, 2:28
PROJECT   = '/content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial'
TEAM_DIR  = f'{PROJECT}/01_Data/Team_AlHada'
FPS       = 1.0          # frames per second to keep (about 310 frames for both clips)
DET_MODEL = 'yolov8s.pt' # COCO-pretrained; detects person, car, motorcycle, bus, truck
DET_CONF  = 0.20
DET_IMGSZ = 1280         # large input: vehicles are small in the wide shot
MIN_PLATE_SIDE = 40      # vehicles whose shorter box side is below this (px) cannot show a readable plate
SEED = 42
# extra regions to blur after review: {'AlHada_CLIP01_t0042.jpg': [(x, y, w, h), ...]}
MANUAL_BLUR = {}
# -------------------------------

## 1. Mount Drive and check inputs

In [ ]:
import os, json, time, glob, random, subprocess, shutil
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

assert os.path.isdir(RAW_DIR), f'Raw folder not found: {RAW_DIR}. Use the KAUST (owner) account.'
assert os.path.isdir(TEAM_DIR), f'Team folder not found: {TEAM_DIR}'
for k, f in CLIPS.items():
    p = Path(RAW_DIR) / f
    assert p.exists(), f'missing {p}'
    print(k, f, f'{p.stat().st_size/1e6:.1f} MB')

TODAY = time.strftime('%Y-%m-%d')
LOCAL_RAW = Path('/content/alhada_raw')       # unblurred, local only, deleted at the end
LOCAL_RAW.mkdir(exist_ok=True)
OUT_DIR = Path(TEAM_DIR) / 'Frames'
OUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)

## 2. Probe the clips (resolution, fps, HDR)

In [ ]:
def probe(path):
    r = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
                        'stream=width,height,r_frame_rate,codec_name,color_transfer,color_primaries,duration:stream_side_data=rotation',
                        '-of', 'json', str(path)], capture_output=True, text=True)
    return json.loads(r.stdout)['streams'][0]

INFO = {}
for k, f in CLIPS.items():
    s = probe(Path(RAW_DIR) / f)
    INFO[k] = s
    print(k, {x: s.get(x) for x in ['codec_name', 'width', 'height', 'r_frame_rate', 'duration', 'color_transfer']})

CLIP01 {'codec_name': 'hevc', 'width': 1920, 'height': 1080, 'r_frame_rate': '30/1', 'duration': '163.678333', 'color_transfer': 'arib-std-b67'}
CLIP02 {'codec_name': 'hevc', 'width': 1920, 'height': 1080, 'r_frame_rate': '30/1', 'duration': '148.170000', 'color_transfer': 'arib-std-b67'}


## 3. Extract frames (local disk only)
HDR (HLG or PQ) iPhone video looks washed out if read as SDR, so it is tone-mapped to BT.709 when ffmpeg supports it; otherwise the plain extraction is used and noted.

In [ ]:
HDR_TRC = {'arib-std-b67', 'smpte2084'}
TONEMAP = ('zscale=t=linear:npl=100,format=gbrpf32le,zscale=p=bt709,'
           'tonemap=tonemap=hable:desat=0,zscale=t=bt709:m=bt709:r=tv,format=yuv420p')
EXTRACT = {}
for k, f in CLIPS.items():
    src = Path(RAW_DIR) / f
    dst = LOCAL_RAW / k; shutil.rmtree(dst, ignore_errors=True); dst.mkdir()
    hdr = INFO[k].get('color_transfer') in HDR_TRC
    vf = f'fps={FPS}' + (',' + TONEMAP if hdr else '')
    cmd = ['ffmpeg', '-v', 'error', '-i', str(src), '-vf', vf, '-q:v', '2', str(dst / f'AlHada_{k}_t%04d.jpg')]
    r = subprocess.run(cmd, capture_output=True, text=True)
    mode = 'tonemapped' if hdr else 'sdr'
    if r.returncode != 0 and hdr:   # zscale missing: fall back to plain extraction
        print(k, 'tone-mapping not available, plain extraction:', r.stderr[-200:])
        r = subprocess.run(['ffmpeg', '-v', 'error', '-i', str(src), '-vf', f'fps={FPS}', '-q:v', '2',
                            str(dst / f'AlHada_{k}_t%04d.jpg')], capture_output=True, text=True)
        mode = 'hdr_not_tonemapped'
    assert r.returncode == 0, r.stderr[-500:]
    n = len(list(dst.glob('*.jpg')))
    EXTRACT[k] = {'frames': n, 'mode': mode}
    print(k, n, 'frames,', mode)

CLIP01 164 frames, tonemapped
CLIP02 148 frames, tonemapped


## 4. Detect people and vehicles

In [ ]:
!pip -q install ultralytics
from ultralytics import YOLO
import numpy as np, pandas as pd, cv2, torch
model = YOLO(DET_MODEL)
KEEP = {0: 'person', 2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}
dev = 0 if torch.cuda.is_available() else 'cpu'
print('device:', dev)

rows = []
frames = sorted(LOCAL_RAW.rglob('*.jpg'))
for i in range(0, len(frames), 16):
    batch = frames[i:i + 16]
    res = model.predict([str(p) for p in batch], conf=DET_CONF, imgsz=DET_IMGSZ, classes=list(KEEP), device=dev, verbose=False)
    for p, r in zip(batch, res):
        for (x1, y1, x2, y2), c, s in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
            rows.append({'frame': p.name, 'clip': p.parent.name, 'cls': KEEP[int(c)], 'conf': float(s),
                         'x': float(x1), 'y': float(y1), 'w': float(x2 - x1), 'h': float(y2 - y1)})
det = pd.DataFrame(rows)
print(len(frames), 'frames,', len(det), 'detections')
display(det.groupby(['clip', 'cls']).size().unstack(fill_value=0))
det['min_side'] = det[['w', 'h']].min(axis=1)
print('vehicle box shorter side (px):')
display(det[det.cls != 'person'].groupby('clip').min_side.describe(percentiles=[.5, .9, .99]).round(1))

device: 0
312 frames, 39 detections


cls     car  person
clip               
CLIP01    7       5
CLIP02   27       0

vehicle box shorter side (px):


        count  mean  std   min   50%   90%   99%   max
clip                                                  
CLIP01    7.0  19.2  2.8  16.5  18.3  22.7  23.3  23.4
CLIP02   27.0  43.4  5.1  35.1  41.9  49.9  52.1  52.2

## 5. Blur and save to the team folder
* **People:** the whole box, strong blur.
* **Vehicles** with shorter side ≥ `MIN_PLATE_SIDE`: the plate zone (bottom 35% of the box, central 70% of its width, padded). Smaller vehicles are left as is: at that size a plate is a few pixels and cannot be read. The distribution above shows how many vehicles are near the threshold.
* **MANUAL_BLUR:** any extra boxes you add after review.

In [ ]:
def blur_region(img, x, y, w, h, k=0.6):
    H, W = img.shape[:2]
    x1, y1 = max(0, int(x)), max(0, int(y)); x2, y2 = min(W, int(x + w)), min(H, int(y + h))
    if x2 - x1 < 2 or y2 - y1 < 2: return 0
    roi = img[y1:y2, x1:x2]
    ks = max(15, int(min(roi.shape[:2]) * k) | 1)
    small = cv2.resize(roi, (max(1, roi.shape[1] // 8), max(1, roi.shape[0] // 8)), interpolation=cv2.INTER_LINEAR)
    img[y1:y2, x1:x2] = cv2.GaussianBlur(cv2.resize(small, (roi.shape[1], roi.shape[0]), interpolation=cv2.INTER_NEAREST), (ks, ks), 0)
    return 1

def plate_zone(r, pad=0.15):
    w, h = r.w, r.h
    zx, zw = r.x + 0.15 * w, 0.70 * w
    zy, zh = r.y + 0.65 * h, 0.35 * h
    return zx - pad * zw, zy - pad * zh, zw * (1 + 2 * pad), zh * (1 + 2 * pad)

manifest = []
for p in frames:
    img = cv2.imread(str(p)); d = det[det.frame == p.name]
    n_people = n_plates = n_small = 0
    for r in d.itertuples():
        if r.cls == 'person':
            n_people += blur_region(img, r.x - 4, r.y - 4, r.w + 8, r.h + 8, k=0.9)
        elif r.min_side >= MIN_PLATE_SIDE:
            n_plates += blur_region(img, *plate_zone(r))
        else:
            n_small += 1
    n_manual = sum(blur_region(img, *b) for b in MANUAL_BLUR.get(p.name, []))
    cv2.imwrite(str(OUT_DIR / p.name), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    manifest.append({'frame': p.name, 'clip': p.parent.name, 't_sec': int(p.stem.split('_t')[-1]) - 1,
                     'width': img.shape[1], 'height': img.shape[0], 'vehicles': int((d.cls != 'person').sum()),
                     'people_blurred': n_people, 'plate_zones_blurred': n_plates, 'small_vehicles_not_blurred': n_small,
                     'manual_blurs': n_manual})
man = pd.DataFrame(manifest)
man.to_csv(Path(TEAM_DIR) / f'{TODAY}_AlHadaFramesManifest_v1.csv', index=False)
display(man.groupby('clip')[['vehicles', 'people_blurred', 'plate_zones_blurred', 'small_vehicles_not_blurred']].sum())
print('saved', len(man), 'blurred frames to', OUT_DIR)

        vehicles  people_blurred  plate_zones_blurred  \
clip                                                    
CLIP01         7               5                    0   
CLIP02        27               0                    0   

        small_vehicles_not_blurred  
clip                                
CLIP01                           7  
CLIP02                          27  

saved 312 blurred frames to /content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial/01_Data/Team_AlHada/Frames


## 6. Review sheet (check by eye)
Top: 8 random blurred frames. Bottom: the 16 **largest** vehicles (where a plate is most likely readable), cropped from the blurred output. If any plate or face is readable, add its box to `MANUAL_BLUR` in cell 0 and re-run cells 5 and 6.

In [ ]:
import matplotlib.pyplot as plt
sample = random.sample(list(man.frame), min(8, len(man)))
fig, ax = plt.subplots(2, 4, figsize=(20, 7))
for a, f in zip(ax.ravel(), sample):
    a.imshow(cv2.cvtColor(cv2.imread(str(OUT_DIR / f)), cv2.COLOR_BGR2RGB)); a.set_title(f, fontsize=8); a.axis('off')
plt.tight_layout(); plt.show()

big = det[det.cls != 'person'].sort_values('min_side', ascending=False).head(16)
fig, ax = plt.subplots(2, 8, figsize=(22, 6))
for a, r in zip(ax.ravel(), big.itertuples()):
    img = cv2.imread(str(OUT_DIR / r.frame)); pad = 10
    crop = img[max(0, int(r.y) - pad):int(r.y + r.h) + pad, max(0, int(r.x) - pad):int(r.x + r.w) + pad]
    a.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)); a.set_title(f'{r.frame[7:]}\n{r.cls} {int(r.min_side)}px', fontsize=7); a.axis('off')
plt.tight_layout(); plt.savefig(f'/content/{TODAY}_AlHadaBlurReview.png', dpi=110); plt.show()

## 7. Summary, and delete the unblurred frames from the Colab disk

In [ ]:
SUMMARY = {
    'created': time.strftime('%Y-%m-%d %H:%M'), 'seed': SEED, 'fps_sampled': FPS,
    'clips': {k: {'file': CLIPS[k], 'width': INFO[k].get('width'), 'height': INFO[k].get('height'),
                  'duration_s': INFO[k].get('duration'), 'codec': INFO[k].get('codec_name'),
                  'color_transfer': INFO[k].get('color_transfer'), **EXTRACT[k]} for k in CLIPS},
    'frames_saved': int(len(man)), 'saved_to': str(OUT_DIR),
    'detector': {'model': DET_MODEL, 'conf': DET_CONF, 'imgsz': DET_IMGSZ},
    'blur': {'people_boxes': int(man.people_blurred.sum()), 'plate_zones': int(man.plate_zones_blurred.sum()),
             'small_vehicles_not_blurred': int(man.small_vehicles_not_blurred.sum()), 'min_plate_side_px': MIN_PLATE_SIDE,
             'manual_boxes': int(man.manual_blurs.sum())},
    'vehicle_min_side_px': det[det.cls != 'person'].min_side.describe(percentiles=[.5, .9, .99]).round(1).to_dict(),
    'privacy': 'raw clips stay in the private owner-only folder; only blurred frames are shared; no GPS/EXIF in frames',
    'use': 'held-out mountain test set only (never train or model selection)',
}
out = f'{PROJECT}/01_Data/{TODAY}_AlHadaFrames_v1.json'
json.dump(SUMMARY, open(out, 'w'), indent=2, default=str)
shutil.rmtree(LOCAL_RAW, ignore_errors=True)   # unblurred frames removed
print('saved', out); print(json.dumps(SUMMARY, indent=1, default=str))

saved /content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial/01_Data/2026-10-19_AlHadaFrames_v1.json
{
 "created": "2026-10-19 17:05",
 "seed": 42,
 "fps_sampled": 1.0,
 "clips": {
  "CLIP01": {
   "file": "<clip 1>",
   "width": 1920,
   "height": 1080,
   "duration_s": "163.678333",
   "codec": "hevc",
   "color_transfer": "arib-std-b67",
   "frames": 164,
   "mode": "tonemapped"
  },
  "CLIP02": {
   "file": "<clip 2>",
   "width": 1920,
   "height": 1080,
   "duration_s": "148.170000",
   "codec": "hevc",
   "color_transfer": "arib-std-b67",
   "frames": 148,
   "mode": "tonemapped"
  }
 },
 "frames_saved": 312,
 "saved_to": "/content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial/01_Data/Team_AlHada/Frames",
 "detector": {
  "model": "yolov8s.pt",
  "conf": 0.2,
  "imgsz": 1280
 },
 "blur": {
  "people_boxes": 5,
  "plate_zones": 0,
  "small_vehicles_not_blurred": 34,
  "min_plate_side_px": 10000,
  "manual_boxes": 0
 },
 "vehicle_min_side_px": {
  "count": 34.0,
  "mean": 38.4